In [ ]:
import sqlite3
import pandas as pd

conn=sqlite3.connect("cricket.db")

tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    """,
    conn
)

print(tables)

conn.close()


In [5]:

import os

os.chdir(r"C:\Users\LENOVO\Downloads\guvi\crizbuzzz")

print(os.getcwd())


C:\Users\LENOVO\Downloads\guvi\crizbuzzz


In [ ]:
import requests
import json
import os
import time
import sys
#import navbar as navbar1
from navbar import convert_batting_stats, convert_bowling_stats

API_KEYS = [
    "8f41e97626msh3345fa983b84672p1d9757jsned2eb13e9264",
    "bca03b80a2mshcc7e09a32e41806p1445d4jsn2a51049c00cc",
    "984ea21795msh215a377894c6679p114480jsn8d51964727a3"
]

current_key_idx = 0

BASE_URL = "https://cricbuzz-cricket2.p.rapidapi.com"

os.makedirs("data", exist_ok=True)

def get_headers():
    return {
        "x-rapidapi-key": API_KEYS[current_key_idx],
        "x-rapidapi-host": "cricbuzz-cricket2.p.rapidapi.com",
        "Content-Type": "application/json"
    }

def get_data(endpoint):
    global current_key_idx
    while current_key_idx < len(API_KEYS):
        try:
            response = requests.get(BASE_URL + endpoint, headers=get_headers())
            if response.status_code == 429:
                response_text = response.text.lower()
                if "exceeded" in response_text or "quota" in response_text:
                    print(f" Key {current_key_idx + 1} exhausted its daily limit!")
                    current_key_idx += 1
                    if current_key_idx < len(API_KEYS):
                        continue  
                else:
                    time.sleep(5)
                    continue
            if response.status_code == 403:
                print(f"\n Key {current_key_idx + 1} is invalid or forbidden.")
                current_key_idx += 1
                if current_key_idx < len(API_KEYS):
                    continue
                else:
                    break

            response.raise_for_status()
            return response.json()

        except Exception as e:
            print(f"Error in {endpoint}: {e}")
            return None
            
    print("\n CRITICAL: All API keys have been exhausted for the day, Please update the API_KEYS list with fresh keys and run again.")
    print(" All fetched data has already been saved to your JSON files.")
    sys.exit(0) 

def save_json(filename, data):
    with open(f"data/{filename}", "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
def load_json(filename):
    filepath = f"data/{filename}"
    if os.path.exists(filepath):
        with open(filepath, "r", encoding="utf-8") as f:
            try:
                return json.load(f)
            except json.JSONDecodeError:
                pass
    return []


ModuleNotFoundError: No module named 'navbar'

In [ ]:
existing_players_list = load_json("players.json")
existing_batting_players = load_json("batting_stats.json")
existing_bowling_players = load_json("bowling_stats.json")
players = {p["player_id"]: p for p in existing_players_list}
batting_stats = {x["player_id"]for x in existing_batting_players}
bowling_stats = {x["player_id"]for x in existing_bowling_players}

In [ ]:
players_updated = 0
for pid, p_data in players.items():
    if "role" not in p_data or not p_data["role"]:
        print(f"Fetching biography for Player ID: {pid}...")
        player_info = get_data(f"/stats/v1/player/{pid}")
        batting_data= get_data(f"/stats/v1/player/{pid}/batting")
        bowling_data= get_data(f"/stats/v1/player/{pid}/bowling")
        
        if batting_data:
            batting_stats.extend(convert_batting_stats(batting_data,pid,p_data["player_name"]))
            save_json("batting_stats.json", batting_stats)

        if bowling_data:
            bowling_stats.extend(convert_bowling_stats(bowling_data,pid,p_data["player_name"]))
            save_json("bowling_stats.json", bowling_stats)

        if player_info:
            players[pid].update({
                "role": player_info.get("role"), "batting_style": player_info.get("battingStyle"),
                "bowling_style": player_info.get("bowlingStyle"), "birth_place": player_info.get("birthPlace"),
                "country": player_info.get("intlTeam")
            })
            save_json("players.json", list(players.values()))
            players_updated += 1

print(f" Downloaded full biographies for {players_updated} new players.")